# Modelagem

* **Abordagem**: **1.** Redes Neurais **2.** Mineração de texto (TF-IDF) **3.** Variáveis Numéricas e Flags **4.** Optuna.

## OUTPUT do modelo

*  **Escore** representa a chance (**probabilidade estatística**) da contestação ser deferida. **Maior escore, maior chance da contestação ser deferida**.
*  Este escore será transformado num **Rating de Admissibilidade de Deferimento**.

## Checklist de avaliação
- [ ] **KS** treino vs teste.
- [ ] **Decis vs taxa do alvo** com ordenação monotônica do escore - base de treino.
- [ ] **Criação do rating de admissibilidade**/ **cutoff** na base de treino e avaliação na base de teste.
- [ ] **Feature importance** das palavras mais relevantes - base de teste. 

In [ ]:
# =============================================================================
# --- INSTALAÇÃO DE DEPENDÊNCIAS DO AMBIENTE ---
# Executar este bloco apenas na primeira configuração do kernel
# Rodar uma vez por dia
# =============================================================================

!pip install shap      # Instala o framework de interpretabilidade e importância de variáveis
!pip install optuna    # Instala o motor de otimização bayesiana para busca de hiperparâmetros da MLP

## A - Imports

In [1]:
#Decis
decis = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

In [2]:
# --- BIBLIOTECAS DE SUPORTE E MANIPULAÇÃO ---
import numpy as np                 # Operações matemáticas e manipulação de arrays/matrizes
import pandas as pd                # Manipulação, limpeza e análise de dados tabulares
import re                          # Expressões regulares para limpeza e padronização de strings
import unicodedata                 # Normalização de caracteres e remoção de acentuação
from scipy import stats            # Funções de estatística avançada e detecção de outliers

# --- PROCESSAMENTO DE LINGUAGEM NATURAL (NLP) ---
import spacy                       # Motor linguístico avançado: Lematização, filtragem gramatical (POS Tagging) e processamento em lote (C-level)

# --- VETORIZAÇÃO E MACHINE LEARNING ---
from sklearn.feature_extraction.text import TfidfVectorizer  # Vetorização estatística por relevância de termos (TF-IDF)
from sklearn.neural_network import MLPClassifier             # Algoritmo de Rede Neural Perceptron Multicamadas
from sklearn.model_selection import train_test_split         # Divisão dos dados entre treino e teste
from sklearn.preprocessing import StandardScaler             # Padronização de recursos (Z-score) para garantir escalas uniformes em variáveis monetárias
import optuna                                                # Framework de otimização bayesiana para busca automática de melhores parâmetros
from sklearn.model_selection import cross_val_score          # Avaliação de performance via validação cruzada (K-Fold) para garantir robustez

# --- COMPOSIÇÃO DE PIPELINES E TRATAMENTO ---
from sklearn.pipeline import Pipeline                 # Automação do fluxo sequencial (Engenharia -> Vetorização -> Modelo)
from sklearn.compose import ColumnTransformer         # Aplicação de tratamentos específicos para diferentes tipos de colunas (Numérica vs Texto)
from sklearn.impute import SimpleImputer              # Tratamento de dados ausentes (Imputação de valores nulos)
from sklearn.preprocessing import FunctionTransformer # Encapsulamento de funções customizadas (ex: Cálculo do Delta) no Pipeline

# --- FEATURE IMPORTANCE ---
import shap                        # Interpretabilidade de modelos complexos e cálculo de importância das variáveis (Shapley Values)

# --- REPRODUTIBILIDADE ---
import joblib                      # Persistência de objetos: utilizado para salvar e carregar o modelo treinado e o pipeline
import json                        # Manipulação de arquivos JSON: utilizado para exportar hiperparâmetros e metadados de configuração

# --- CONFIGURAÇÕES DO AMBIENTE ---
import warnings
warnings.filterwarnings("ignore")  # Suprime alertas de sistema para manter o output limpo
pd.set_option('display.max_columns', None) # Garante a visualização de todas as colunas do DataFrame

## B - Suporte a modelagem

### Saneamento do texto

In [3]:
# 1. FUNÇÃO DE SANEAMENTO (PRE-PROCESSING)
def pre_processar_texto(texto):
    if not isinstance(texto, str):
        return ""
    
    # A. Tudo para minúsculo
    texto = texto.lower()
    
    # B. Remoção de acentos (Normalização Unicode)
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    
    # C. Remoção de caracteres especiais, hifens e pontuação
    # Mantém apenas letras e espaços
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    
    # D. Remoção de espaços extras
    texto = " ".join(texto.split())
    
    return texto

### Stopwords

In [4]:
# DEFINIÇÃO DE STOPWORDS EM PORTUGUÊS
stopwords_pt = [
    # Artigos e preposições originais
    'a', 'ao', 'aos', 'as', 'com', 'da', 'das', 'de', 'do', 'dos', 'e', 'em', 'que', 'se', 'quando', 'for',
    'um', 'uma', 'os', 'no', 'na', 'nos', 'nas', 'por', 'para', 'pelo', 'pela', 'pois',
    
    # Contrações adicionais
    'à', 'às', 'dum', 'duma', 'num', 'numa', 'neste', 'nesta', 'nesse', 'nessa', 'naquele', 'naquela',
    'dele', 'dela', 'deles', 'delas', 'nele', 'nela', 'neles', 'nelas', 'desse', 'dessa', 'deste', 'desta',
    
    # Pronomes
    'eu', 'ele', 'ela', 'nós', 'eles', 'elas', 'me', 'te', 'lhe', 'lhes',
    'meu', 'minha', 'meus', 'minhas', 'seu', 'sua', 'seus', 'suas', 'nosso', 'nossa',
    'este', 'esta', 'estes', 'estas', 'esse', 'essa', 'esses', 'essas', 'isso', 'isto', 'aquilo',
    
    # Conjunções e Adverbios genéricos
    'como', 'mas', 'porém', 'ou', 'porque', 'até', 'entre', 'sem', 'sobre', 'sob', 'desde', 'já', 'nem', 
    'logo', 'portanto', 'também', 'apenas', 'só', 'assim', 'ainda', 'mesmo',
    
    # Verbos auxiliares e de ligação de altíssima frequência 
    'é', 'são', 'foi', 'foram', 'era', 'está', 'estão', 'estava', 'ser', 'ter', 'tem', 'têm', 'tinha', 'há', 'havia',

    # Sefaz
    'valor', 'valores', 'icms', 'nota', 'notas', 'fiscal', 'fiscais', 'cfop', 'mercadoria', 'mercadorias', 'operação', 
    'operações', 'artigo', 'lei', 'conforme', 'nos', 'termos', 'seja', 'devido', 'pago', 'pagamento', 
    'recolhimento', 'referente', 'mencionado', 'citado', 'anexo', 'doc', 'docs',
    
    # Formalismos de Petição (Polidez que engana o modelo)
    'vossa', 'senhoria', 'atenciosamente', 'cordialmente', 'ilustríssimo', 'douto', 'julgador', 'recorrente', 
    'veio', 'vem', 'requer', 'solicita','diante', 'exposto', 'pede', 'deferimento', 'tempestivo', 'legal', 'processo'
]

### Faixa de valores vs ALVO

* Aplicar esta função sob uma perspectiva bivariada: variável numérica em conjunto com o ALVO (variável resposta).

In [5]:
def taxa_por_decil(df, col_escore, col_target, percentis=None, n_decis=10):
    """
    Calcula para cada grupo (decil ou percentil personalizado):
      - range: mínimo e máximo do escore no grupo
      - volumetria: número de observações
      - taxa do evento: proporção da variável categórica (média)

    Parâmetros:
    - df: DataFrame
    - col_escore: nome da coluna numérica (ex: escore)
    - col_target: nome da coluna binária (0/1) ou categórica para calcular a taxa
    - percentis: lista de percentis (ex: [0, 0.1, 0.2, ..., 1]) - se None, usa n_decis
    - n_decis: número de grupos de igual tamanho (usado se percentis for None)

    Retorna:
    - DataFrame com as colunas:
        - decil (ou faixa)
        - min_escore, max_escore
        - volumetria
        - taxa_evento
    """
    # Se percentis não for fornecido, cria decis igualmente espaçados
    if percentis is None:
        percentis = np.linspace(0, 1, n_decis + 1)
    
    # Calcula os valores dos percentis na coluna de escore
    limites = df[col_escore].quantile(percentis).values
    
    # Remove limites duplicados (caso haja pouca variação nos dados)
    limites = np.unique(limites)
    if len(limites) < 2:
        raise ValueError("Não foi possível criar múltiplos grupos. Verifique a variabilidade do escore.")
    
    # Cria a coluna de faixas
    df_temp = df.copy()
    df_temp['faixa'] = pd.cut(df_temp[col_escore], bins=limites, include_lowest=True, right=True)
    
    # Agrupa por faixa e calcula as estatísticas
    resultado = df_temp.groupby('faixa').agg(
        valor_minimo =(col_escore, 'min'),
        valor_maximo =(col_escore, 'max'),
        volumetria=(col_target, 'count'),
        taxa_evento=(col_target, 'mean')
    ).reset_index()
    
    # Renomear a coluna 'faixa' para 'decil' (ou manter como faixa)
    resultado = resultado.rename(columns={'faixa': 'range_valor'})
    
    # Arredondar a taxa para 4 casas decimais (opcional)
    resultado['taxa_evento'] = resultado['taxa_evento'].round(4)
    
    return resultado

### Estatística KS 

* Performance do modelo.

In [6]:
def calcular_ks_2samp(df, alvo, escore):
    bons_escore = df.loc[df[alvo] == 0, escore].rename('bons')
    maus_escore = df.loc[df[alvo] == 1, escore].rename('maus')
    return stats.ks_2samp(bons_escore, maus_escore).statistic

# 1 - Base de Modelagem

In [7]:
df = pd.read_csv('base_modelagem.csv', engine='python', encoding='utf-8', sep=';', on_bad_lines='skip')
#Visualização
df.head(3)

,nome_razao,inscricao,periodo_fiscal,registro_nota,nf_sq_registro,contestacao,regime_contrib.,segmento_economico,cod_motivo,procontenf_tx_justificativa,valor_cobrado,valor_sugerido_contrib,resultado
0,DARLEY A DE HOLANDA ME,54822831,202412,012.6.097787376-7,97787376,202400001027919522,05-SIMPLES,007-INDUSTRIA,255,AQUISIÇÃO DE UMA MÁQUINA INJETORA PARA UTILIZA...,0.00,0.0,1
1,DARLEY A DE HOLANDA ME,54822831,202412,012.6.103977779-0,103977779,202400001027946252,05-SIMPLES,007-INDUSTRIA,255,AQUISIÇÃO DE UMA PARTE DE UMA MÁQUINA INJETORA...,2805.82,0.0,0
2,MASSA FEST INDUSTRIA DE MASSAS ALIMENTICIAS LTDA,73690856,202412,012.6.124667047-9,124667047,202400001015709285,01-NORMAL,008-INDUSTRIA DE ALIMENTOS,261,MERCADORIA DESTINADA A INDUSTRIALIZAÇÃO (MATER...,2151.22,0.0,0


In [8]:
#Informações do data frame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138860 entries, 0 to 138859
Data columns (total 13 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   nome_razao                   138860 non-null  object 
 1   inscricao                    138860 non-null  int64  
 2   periodo_fiscal               138860 non-null  int64  
 3   registro_nota                138860 non-null  object 
 4   nf_sq_registro               138860 non-null  int64  
 5   contestacao                  138860 non-null  int64  
 6   regime_contrib.              138860 non-null  object 
 7   segmento_economico           138860 non-null  object 
 8   cod_motivo                   138860 non-null  int64  
 9   procontenf_tx_justificativa  127470 non-null  object 
 10  valor_cobrado                138860 non-null  float64
 11  valor_sugerido_contrib       138860 non-null  float64
 12  resultado                    138860 non-null  int64  
dtyp

In [9]:
#Volumetria
volumetria = df.shape[0] 
print(f"Volumetria de modelagem: {volumetria}") #138.860

Volumetria de modelagem: 138860


In [10]:
#Taxa de contestação deferida
taxa_deferimento = round(100 * df['resultado'].value_counts(normalize=True).loc[1], 2)
print(f"Taxa de aceite: {taxa_deferimento}%") #52.09%

Taxa de aceite: 52.09%


# 2 - Orquestrador de Transformação de Dados

Este bloco centraliza a engenharia de atributos e a vetorização matemática. O `ColumnTransformer` atua como o motor principal, roteando cada tipo de variável para o seu respectivo funil de pré-processamento antes de alimentar a Rede Neural (MLP).

* **Variáveis Contínuas:** Recebem imputação de nulos e padronização de escala (Z-score).
* **Variáveis de Texto:** Passam pelo motor TF-IDF com amortecimento logarítmico (`sublinear_tf=True`) para mitigação de repetições anômalas.
* **Flags Binárias (Target-Guided Binning):** Seguem diretamente (via `passthrough`) para preservar os agrupamentos de risco originais.

## Criação de Variáveis

In [11]:
#Variáveis numéricas
##Variável delta financeiro
df['delta_financeiro'] = df['valor_cobrado'].fillna(0) - df['valor_sugerido_contrib'].fillna(0)
##Variável total de palavras
df['total_palavras'] = df['procontenf_tx_justificativa'].fillna("").apply(lambda x: len(str(x).split()))

In [12]:
#Variáveis flag
##código de motivo
df['flag_cd_214'] = np.where(df['cod_motivo'] == 214, 1,0)
df['flag_cd_232'] = np.where(df['cod_motivo'] == 232, 1,0)
df['flag_cd_taxa_alta'] = df['cod_motivo'].isin([228, 346, 343, 272, 211]).astype(int)
df['flag_cd_taxa_baixa'] = np.where(df['cod_motivo'] == 265, 1,0)
##delta financeiro
df['flag_df_positivo'] = np.where(df['delta_financeiro']>0, 1, 0)
##regime de contribuição
df['flag_rc_simples'] = np.where(df['regime_contrib.'] == '05-SIMPLES', 1,0)
##segmento econômico
df['flag_se_taxa_alta'] = np.where((df['segmento_economico'] == '018-VEICULOS') | (df['segmento_economico'] == '012-SUPERMERCADOS'), 1, 0)
df['flag_se_taxa_baixa'] = np.where(df['segmento_economico'] == '013-TECIDOS', 1,0)
##valor cobrado
df['flag_vc_ate_100'] = np.where(df['valor_cobrado']<=100, 1, 0)

In [13]:
#Variável de texto
df['flag_justificativa_ausente'] = df['procontenf_tx_justificativa'].isna().astype(int) #Indica se o contribuiente escreveu algo no campo da justificativa
df['justificativa_limpa'] = df['procontenf_tx_justificativa'].fillna('sem_justificativa_informada').apply(pre_processar_texto) #Feature Textual Principal

## Pipeline Automatizado de Extração de Atributos

In [14]:
# 1. Definição das colunas numéricas 
colunas_numericas = ['valor_cobrado', 'valor_sugerido_contrib', 'total_palavras', 'delta_financeiro']

# 2. Definição das colunas de Flags (Variáveis Binárias de Negócio / Target-Guided Binning)
colunas_flags = [
    'flag_justificativa_ausente',
    'flag_cd_214',
    'flag_cd_232',
    'flag_cd_taxa_alta',
    'flag_cd_taxa_baixa',
    'flag_df_positivo',
    'flag_rc_simples',
    'flag_se_taxa_alta',
    'flag_se_taxa_baixa',
    'flag_vc_ate_100'
]

# 3. Pipeline Numérico: Trata nulos e aplica normalização (Z-score)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

# 4. Pipeline de Texto: Vetorização TF-IDF
text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),        # Mantemos bigramas para pegar "erro_digitação", etc.
        max_features=1500,         # Aumentamos para 1500 para captar termos de mérito mais raros
        stop_words=stopwords_pt,   # A nova lista robusta
        min_df=5,                  # Filtra erros de digitação
        max_df=0.85,               # NOVO: Ignora termos que aparecem em mais de 85% da base
        sublinear_tf=True          # Amortece textos muito longos e repetitivos
    ))
])

# 5. Orquestrador Final (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, colunas_numericas), # Aplica em todas as numéricas
        ('text', text_transformer, 'justificativa_limpa'),
        ('flag', 'passthrough', colunas_flags) #passthrough significa "apenas passe adiante"
    ],
    remainder='drop' # Descarta o restante das colunas
)

# 3 - Modelagem

## Divisão dos dados 
* Treino e Teste

In [15]:
# Separação 
X = df.drop(columns=['resultado'])
y = df['resultado']
perc_teste = 0.25

# Bases de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= perc_teste, random_state=42, stratify=y)

## Identificar quais as palavras mais relevantes na abordagem TF IDF

### Etapas:

* Aplicar na base treino.
* Filtro de relevância.
* Termos-chave da contestação.

In [16]:
# 1. Ajustar o preprocessor apenas nos dados de treino (evita data leakage)
preprocessor.fit(X_train)

# 2. Acessar o TfidfVectorizer dentro do ColumnTransformer
# O índice [1] refere-se ao segundo transformer definido: ('text', text_transformer, 'justificativa_limpa')
tfidf_extractor = preprocessor.named_transformers_['text'].named_steps['tfidf']

# 3. Extrair os nomes das features (palavras) e seus respectivos pesos IDF
palavras = tfidf_extractor.get_feature_names_out()
pesos_idf = tfidf_extractor.idf_

# 4. Criar DataFrame para análise
df_ranking_tfidf = pd.DataFrame({
    'termo': palavras,
    'importancia_idf': pesos_idf
})

# 5. Ordenar: Maior IDF = Termo mais raro e potencialmente mais explicativo
top_20_palavras = df_ranking_tfidf.sort_values(by='importancia_idf', ascending=False).head(20)

print("🏛️ Top 20 Termos mais relevantes (TF-IDF) na Base de Treino:")
print(top_20_palavras)

🏛️ Top 20 Termos mais relevantes (TF-IDF) na Base de Treino:
                       termo  importancia_idf
401          corrigido pagar         8.621723
999             nf corrigido         8.621723
83            antecipacao nf         8.602305
1055       pagar antecipacao         8.602305
400                corrigido         8.042690
1040           ordem servico         7.478375
772               impugnante         7.423650
1322  servicos farmaceuticos         7.366163
1065          pauta aliquota         7.182911
425                dec vigor         7.155386
1487            vigor partir         7.155386
885                junho dec         7.146377
1129       previstos decreto         7.124203
50                   algodao         7.119827
72             ambos decreto         7.068752
1161              produzidos         7.064611
662            farmaceuticos         6.985205
430              decreto dec         6.966300
457             deslocamento         6.958838
85                a

## Aplicação do algoritmo

**Manual de operações**

In [17]:
def objective(trial):
    # 1. Define dinamicamente o número de camadas ocultas (profundidade da rede)
    n_layers = trial.suggest_int('n_layers', 1, 3) #O Optuna sorteia um número inteiro entre 1 e 3.
    
    # 2. Define a quantidade de neurônios para cada camada sugerida acima (largura da rede)
    layers = []
    for i in range(n_layers):
        layers.append(trial.suggest_int(f'n_units_l{i}', 32, 256)) #Para cada camada, o Optuna sorteia a quantidade de neurônios entre 32 e 256.
    
    # 3. Dicionário de hiperparâmetros a serem testados pelo algoritmo bayesiano do Optuna
    params = {
        'hidden_layer_sizes': tuple(layers),
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']), # Função de ativação dos neurônios
        'solver': 'adam', # Algoritmo de otimização de pesos (padrão para grandes datasets)
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True), # Penalidade L2 (evita que o modelo "decore" os dados)
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True), # Velocidade de aprendizado inicial
        'max_iter': 200,          # Sugestão: Reduzido de 500 para 200
        'early_stopping': True,   # Trava de aceleração ativada
        'random_state': 42 # Garante reprodutibilidade dos resultados
    }
    
    # 4. Integra o pré-processamento validado anteriormente com a rede neural sugerida
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('mlp', MLPClassifier(**params))
    ])
    
    # 5. Executa a validação cruzada para garantir que o resultado seja estável em diferentes fatias dos dados
    # cv=3 indica que os dados de treino serão divididos em 3 subconjuntos (folds)
    # n_jobs=-1 utiliza todos os núcleos do processador para acelerar o treino
    # scoring='precision' foca em minimizar falsos positivos — evita deferir contestações indevidas
    score = cross_val_score(clf, X_train, y_train, n_jobs=-1, cv=3, scoring='precision').mean() #aqui onde tem o cross validation/ o total de folds
    
    return score # Retorna a média da métrica precison para o Optuna decidir o próximo palpite

**Execução**

In [18]:
# --- EXECUÇÃO DA OTIMIZAÇÃO BAYESIANA ---

# 1. Instancia o estudo focado em maximizar a métrica de Precision
# O Optuna usará algoritmos de probabilidade para encontrar os melhores valores
study = optuna.create_study(direction='maximize')

# 2. Inicia as tentativas (trials)
# Cada trial treina o modelo 3 vezes (devido ao cv=3) e reporta a média
study.optimize(objective, n_trials=20)

# ----------------------------------------------------------------------------------------------------
#Se eu quiser reprodutibilidade
#from optuna.samplers import TPESampler
#sampler = TPESampler(seed=42) # O número 42 garante que os "chutes" sejam sempre os mesmos
#study = optuna.create_study(direction='maximize', sampler=sampler)
# ----------------------------------------------------------------------------------------------------

# ---------------------------------------------------------
# 3. RELATÓRIO EXECUTIVO DE HIPERPARÂMETROS
# ---------------------------------------------------------
print("-" * 50)
print("🎯 RESULTADO DA OTIMIZAÇÃO DE HIPERPARÂMETROS")
print("-" * 50)
print(f"Melhor Precision alcançada (Média CV): {study.best_value:.4f}")
print("\nConfiguração Ótima para a Rede Neural:")

for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f" 🔹 {key:20}: {value:.6f}")
    else:
        print(f" 🔹 {key:20}: {value}")
print("-" * 50)

[I 2026-03-26 12:16:28,835] A new study created in memory with name: no-name-65095459-4275-4d5d-9937-38972988bad1
[I 2026-03-26 12:18:16,819] Trial 0 finished with value: 0.9676733588797669 and parameters: {'n_layers': 1, 'n_units_l0': 192, 'activation': 'relu', 'alpha': 9.189308823680971e-05, 'learning_rate_init': 0.000515504007545729}. Best is trial 0 with value: 0.9676733588797669.
[I 2026-03-26 12:21:39,857] Trial 1 finished with value: 0.9660494095840123 and parameters: {'n_layers': 2, 'n_units_l0': 245, 'n_units_l1': 194, 'activation': 'tanh', 'alpha': 0.001459065762083058, 'learning_rate_init': 0.0032513911791457557}. Best is trial 0 with value: 0.9676733588797669.
[I 2026-03-26 12:25:43,746] Trial 2 finished with value: 0.9565550562898352 and parameters: {'n_layers': 2, 'n_units_l0': 184, 'n_units_l1': 132, 'activation': 'tanh', 'alpha': 0.008909121725104897, 'learning_rate_init': 0.00022698894342443904}. Best is trial 0 with value: 0.9676733588797669.
[I 2026-03-26 12:29:11,57

--------------------------------------------------
🎯 RESULTADO DA OTIMIZAÇÃO DE HIPERPARÂMETROS
--------------------------------------------------
Melhor Precision alcançada (Média CV): 0.9677

Configuração Ótima para a Rede Neural:
 🔹 n_layers            : 1
 🔹 n_units_l0          : 192
 🔹 activation          : relu
 🔹 alpha               : 0.000092
 🔹 learning_rate_init  : 0.000516
--------------------------------------------------


**Escoragem**

* Bases de treino, teste e full.

In [19]:
# 1. Configuração dos parâmetros extraídos da sua otimização
best_params = {
    'hidden_layer_sizes': (192,),    # 1 camada oculta com 192 neurônios
    'activation': 'relu',            # Função de ativação selecionada
    'alpha': 0.000092,               # Regularização L2
    'learning_rate_init': 0.000516,  # Taxa de aprendizado
    'solver': 'adam',
    'early_stopping': True,          # Mantemos a trava de segurança
    'random_state': 42,
    'max_iter': 200
}

# 2. Instanciação do Pipeline Final
modelo_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('mlp', MLPClassifier(**best_params))
]) #modelo criado

# 3. Treinamento na base de treino (X_train)
modelo_final.fit(X_train, y_train) 

# 4. Função para estruturar os DataFrames com Escore e Alvo
def escorar_base(modelo, X, y):
    df_escorado = X.copy()
    # Probabilidade da classe 1 (Procedente)
    df_escorado['escore_procedencia'] = modelo.predict_proba(X)[:, 1]
    df_escorado['resultado'] = y
    return df_escorado

# 5. Geração dos três DataFrames solicitados
df_treino_escorado   = escorar_base(modelo_final, X_train, y_train)
df_teste_escorado    = escorar_base(modelo_final, X_test, y_test)
df_completa_escorada = escorar_base(modelo_final, X, y)

# --- VALIDAÇÃO EXECUTIVA ---
print(f"Check: Base Completa Escorada - {df_completa_escorada.shape[0]} registros.")
display(df_completa_escorada[['escore_procedencia', 'resultado']].head())

Check: Base Completa Escorada - 138860 registros.


,escore_procedencia,resultado
0,0.997996,1
1,0.000933,0
2,0.006774,0
3,0.000471,0
4,0.998865,1


# 4 - Avaliação

## KS das bases

In [20]:
# =========================================================
# --- CÁLCULO DA ESTATÍSTICA KS (KOLMOGOROV-SMIRNOV) ---
# =========================================================

# Utilizando a função calcular_ks_2samp definida anteriormente
ks_treino   = calcular_ks_2samp(df_treino_escorado, 'resultado', 'escore_procedencia')
ks_teste    = calcular_ks_2samp(df_teste_escorado, 'resultado', 'escore_procedencia')
ks_completa = calcular_ks_2samp(df_completa_escorada, 'resultado', 'escore_procedencia')

print("-" * 50)
print("📊 RESULTADO DO KS (KOLMOGOROV-SMIRNOV)")
print("-" * 50)
print(f" 🔹 KS Treino   : {ks_treino:.4f} ({(ks_treino*100):.2f})")
print(f" 🔹 KS Teste    : {ks_teste:.4f} ({(ks_teste*100):.2f})")
print(f" 🔹 KS Completa : {ks_completa:.4f} ({(ks_completa*100):.2f})")
print("-" * 50)

--------------------------------------------------
📊 RESULTADO DO KS (KOLMOGOROV-SMIRNOV)
--------------------------------------------------
 🔹 KS Treino   : 0.9662 (96.62)
 🔹 KS Teste    : 0.9404 (94.04)
 🔹 KS Completa : 0.9596 (95.96)
--------------------------------------------------


## Base de Treino

In [21]:
df_treino_escorado.head(3)

,nome_razao,inscricao,periodo_fiscal,registro_nota,nf_sq_registro,contestacao,regime_contrib.,segmento_economico,cod_motivo,procontenf_tx_justificativa,valor_cobrado,valor_sugerido_contrib,delta_financeiro,total_palavras,flag_cd_214,flag_cd_232,flag_cd_taxa_alta,flag_cd_taxa_baixa,flag_df_positivo,flag_rc_simples,flag_se_taxa_alta,flag_se_taxa_baixa,flag_vc_ate_100,flag_justificativa_ausente,justificativa_limpa,escore_procedencia,resultado
16007,MUNDIAL ATACADO DE ARTIGOS DE ARMARINHO LTDA,113887060,202410,012.6.296382949-0,296382949,202400000980847709,05-SIMPLES,013-TECIDOS,214,SOLICITA RECALCULO,409.01,0.0,409.01,2,1,0,0,0,1,1,0,1,0,0,solicita recalculo,9.978515e-03,0
116000,MCA ALIMENTOS LTDA,110089030,202504,012.6.329972549-5,329972549,202500000377641723,01-NORMAL,012-SUPERMERCADOS,214,O EXTRATO-FRONTEIRAS NÃO POSSUI NATUREZA JURÍD...,3552.03,0.0,3552.03,270,1,0,0,0,1,0,1,0,0,0,o extrato fronteiras nao possui natureza jurid...,2.087658e-07,0
100911,SILVESTRE SOLUCOES COMERCIO VAREJISTA EM INFOR...,107226456,202507,012.6.250567287-0,250567287,202500000829336540,01-NORMAL,017-VAREJO,270,DISPENSADO DA ANTECIPAÇÃO CONTRIBUINTE BENEFIC...,0.00,0.0,0.00,23,0,0,0,0,0,0,0,0,1,0,dispensado da antecipacao contribuinte benefic...,9.999826e-01,1


In [22]:
#Volumetria
df_treino_escorado.shape[0] #104.145

104145

In [23]:
#Distribuição dos escores
#Análise descritiva de alguma variável numérica
df_treino_escorado['escore_procedencia'].describe(percentiles=decis)

count    1.041450e+05
mean     5.205872e-01
std      4.839574e-01
min      2.815912e-15
10%      2.320350e-06
20%      6.508709e-05
30%      1.045988e-03
40%      1.474228e-02
50%      8.524674e-01
60%      9.964605e-01
70%      9.996980e-01
80%      9.999640e-01
90%      9.999968e-01
max      1.000000e+00
Name: escore_procedencia, dtype: float64

### Critérios de escolha dos cutoffs para o Rating

Os limiares foram definidos com base na tabela de decis da base de treino:

- **≤ 0.04 → `contestacao_indeferida`**: corresponde aproximadamente ao 4º decil inferior,
  onde a taxa de deferimento é inferior a 0.5%. Risco praticamente nulo de deferir indevidamente. (`Nota: Poderia baixar para 0.02`).

- **> 0.04 e ≤ 0.9 → `contestacao_para_analise`**: zona de incerteza do modelo (~10% da base). **Requer revisão humana**.

- **> 0.9 → `contestacao_deferida`**: modelo com alta confiança de deferimento.
  Taxa observada de ~98% na base de teste.

In [24]:
#Faixa de escore vs resultado
taxa_por_decil(df_treino_escorado,  'escore_procedencia', 'resultado', percentis=None, n_decis=10)

,range_valor,valor_minimo,valor_maximo,volumetria,taxa_evento
0,"(-9.9999999718409e-07, 2.32035e-06]",2.815912e-15,0.000002,10415,0.0001
1,"(2.32035e-06, 6.50871e-05]",2.320379e-06,0.000065,10414,0.0003
2,"(6.50871e-05, 0.00104599]",6.509126e-05,0.001046,10415,0.0005
3,"(0.00104599, 0.0147423]",1.046311e-03,0.014741,10414,0.0050
4,"(0.0147423, 0.852467]",1.474285e-02,0.852467,10415,0.2359
5,"(0.852467, 0.996461]",8.526791e-01,0.996460,10414,0.9693
6,"(0.996461, 0.999698]",9.964609e-01,0.999698,10445,0.9989
7,"(0.999698, 0.999964]",9.996981e-01,0.999964,10384,0.9998
8,"(0.999964, 0.999997]",9.999640e-01,0.999997,10415,0.9997
9,"(0.999997, 1.0]",9.999968e-01,1.000000,10414,0.9999


**Desenvolvimento de estratégias**

In [35]:
def aplicar_rating(df):
    # 1. Definição das Condições (Lógica de Negócio)
    condicoes = [
        (df['escore_procedencia'] <= 0.04),
        (df['escore_procedencia'] > 0.04) & (df['escore_procedencia'] <= 0.9),
        (df['escore_procedencia'] > 0.9)
    ]
    
    # 2. Definição dos Retornos (Strings)
    categorias = [
        'contestacao_indeferida', 
        'contestacao_para_analise', 
        'contestacao_deferida'
    ]
    
    # 3. Aplicação com Default em String para evitar conflito de DType
    # O parâmetro default='outros' garante que o NumPy veja apenas Strings
    df['rating'] = np.select(condicoes, categorias, default='fora_do_range')
    
    return df

# Re-executando a aplicação
df_treino_escorado   = aplicar_rating(df_treino_escorado)
df_teste_escorado    = aplicar_rating(df_teste_escorado)
df_completa_escorada = aplicar_rating(df_completa_escorada)

# Validando o sucesso
print("✅ Rating aplicado com sucesso em todas as bases.")
print(df_completa_escorada['rating'].value_counts())

✅ Rating aplicado com sucesso em todas as bases.
rating
contestacao_deferida        67513
contestacao_indeferida      58386
contestacao_para_analise    12961
Name: count, dtype: int64


## Base de Teste

In [36]:
df_teste_escorado.shape[0] #34715

34715

In [37]:
# Criar tabela com as colunas solicitadas
tabela_analise_0 = df_teste_escorado.groupby('rating').agg(
    total_processos_analisados=('rating', 'count'),
    total_deferidos=('resultado', 'sum')
).reset_index()

# Calcular percentual de processos analisados (frequência relativa)
tabela_analise_0['percentual_processos_analisados'] = (tabela_analise_0['total_processos_analisados'] / len(df_teste_escorado) * 100).round(2)

# Calcular taxa de deferimento
tabela_analise_0['taxa_deferimento'] = (tabela_analise_0['total_deferidos'] / tabela_analise_0['total_processos_analisados'] * 100).round(2)

# Selecionar apenas as colunas desejadas
tabela_analise_0 = tabela_analise_0[['rating', 
                                  'total_processos_analisados', 
                                  'percentual_processos_analisados', 
                                  'taxa_deferimento']]

# Ordenar por total de processos (do maior para o menor)
tabela_analise_0 = tabela_analise_0.sort_values('total_processos_analisados', ascending=False).reset_index(drop=True)

#Visualização
tabela_analise_0.head(3)

,rating,total_processos_analisados,percentual_processos_analisados,taxa_deferimento
0,contestacao_deferida,16834,48.49,98.60
1,contestacao_indeferida,14457,41.64,0.68
2,contestacao_para_analise,3424,9.86,40.54


**Feature Importance**

In [28]:
# ---------------------------------------------------------
# 1. PREPARAÇÃO DOS DADOS DE TESTE
# ---------------------------------------------------------
# Transformamos a base de teste (que o modelo nunca viu) para o formato numérico/vetorial
X_test_transformed = modelo_final.named_steps['preprocessor'].transform(X_test)

# Conversão necessária caso o TF-IDF gere matrizes esparsas
if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

# ---------------------------------------------------------
# 2. INICIALIZAÇÃO DO EXPLICAÇÃO (SHAP)
# ---------------------------------------------------------
# O KernelExplainer "aprende" o comportamento do modelo.
# Usamos shap.sample para criar um resumo da base (background) e acelerar o cálculo.
explainer = shap.KernelExplainer(
    modelo_final.named_steps['mlp'].predict_proba, 
    shap.sample(X_test_transformed, 50) # Amostra de referência
)

# ---------------------------------------------------------
# 3. CÁLCULO DOS VALORES SHAP (A "AUDITORIA" DOS TESTES)
# ---------------------------------------------------------
# ATENÇÃO: É aqui que definimos que estamos usando a BASE DE TESTE.
# O [:200] aumenta a robustez estatística para a diretoria.
# 1. Definimos um número maior para dar robustez
tamanho_amostra = 200 

# 2. Pegamos sempre os mesmos registros para o relatório não mudar
amostra_teste = X_test_transformed[:tamanho_amostra] 

# 3. O restante do código segue igual
shap_values = explainer.shap_values(amostra_teste)

# Em modelos binários, o SHAP gera uma lista. 
# Selecionamos a classe [1] (geralmente "Improcedente" ou "Fraude")
if isinstance(shap_values, list):
    current_shap_values = shap_values[1]
else:
    current_shap_values = shap_values

# ---------------------------------------------------------
# 4. CONSTRUÇÃO DA TABELA DE IMPACTO (FEATURE IMPORTANCE)
# ---------------------------------------------------------
# Recuperamos os nomes das colunas (Variáveis + Palavras do Texto)
feature_names = modelo_final.named_steps['preprocessor'].get_feature_names_out()

# Calculamos a força (média absoluta) do impacto de cada variável
importances = np.abs(np.atleast_2d(current_shap_values)).mean(axis=0).flatten()

# Alinhamos nomes e pesos em um DataFrame consolidado
df_geral = pd.DataFrame(list(zip(feature_names, importances)), 
                        columns=['Variável', 'Impacto Médio (SHAP)'])

# Limpeza e Organização para o Relatório
df_geral = df_geral.sort_values(by='Impacto Médio (SHAP)', ascending=False)
df_geral['Variável'] = df_geral['Variável'].str.replace('text__', '').str.replace('num__', '').str.replace('remainder__', '')

# ---------------------------------------------------------
# 5. RESULTADO FINAL
# ---------------------------------------------------------
print("\n" + "="*50)
print("🎯 RELATÓRIO DE IMPORTÂNCIA: BASE DE TESTE (N=200)")
print("="*50)
print(df_geral.head(10).to_string(index=False))
print("="*50)

  0%|          | 0/200 [00:00<?, ?it/s]


🎯 RELATÓRIO DE IMPORTÂNCIA: BASE DE TESTE (N=100)
                        Variável  Impacto Médio (SHAP)
                delta_financeiro              0.021185
                  total_palavras              0.021185
                   valor_cobrado              0.011854
          valor_sugerido_contrib              0.011854
                          acesso              0.010765
                      acessorias              0.010765
flag__flag_justificativa_ausente              0.004795
               flag__flag_cd_214              0.004795
                        pe forma              0.003710
                           pecas              0.003710


# 5 - Salvar o pipeline

In [29]:
# Definindo o nome do arquivo com a métrica para facilitar a gestão de versões
filename = 'versao_2.pkl'

# Salvando o pipeline (inclui ColumnTransformer + MLP)
joblib.dump(modelo_final, filename)

print(f"✅ Pipeline salvo com sucesso: {filename}")

✅ Pipeline salvo com sucesso: versao_2.pkl


In [30]:
# 1. Exportar histórico de todas as 20 tentativas para auditoria
df_trials = study.trials_dataframe()
df_trials.to_csv('log_experimento_optuna.csv', index=False)

# 2. Salvar apenas os melhores hiperparâmetros (os vencedores)
best_params_export = study.best_params
best_params_export['best_value_precision'] = study.best_value

with open('best_params_executivo.json', 'w') as f:
    json.dump(best_params_export, f, indent=4)

print("✅ Metadados do experimento exportados para CSV e JSON.")

✅ Metadados do experimento exportados para CSV e JSON.


# 6 - Exportar os resultados

In [38]:
#Base de Treino
df_treino_escorado.to_csv('resultado_escoragem_treino.csv', index=False, sep=';', encoding='utf-8-sig')
#Base de Teste
df_teste_escorado.to_csv('resultado_escoragem_teste.csv', index=False, sep=';', encoding='utf-8-sig')
#Base Full
df_completa_escorada.to_csv('resultado_escoragem_full.csv', index=False, sep=';', encoding='utf-8-sig')